# End-to-end (PyTorch): entrenamiento multi-`k` en Google Colab

Este notebook:

1. Elige **5 valores de `k`** representativos con `k < n` (sin Sionna: el E2E no usa Polar). En la sección opcional del híbrido se filtran los `k` admisibles para Polar 5G.
2. Entrena **una red por cada par** \((k, \rho)\) de forma **secuencial**: termina todos los SNR para un `k` y pasa al siguiente.
3. Guarda checkpoints con nombre fijo: `results/checkpoints/e2e_k{k}_n100_rho{rho}.pt`.
4. Genera las **mismas curvas** que el híbrido (`P_FA`, `P_MD`, `P_IE`, `P_global` vs `R`) vía `run_curves_for_n`.
5. (Opcional) Entrena el **híbrido** solo en esos mismos `k` y una **figura comparativa**.

## 0. Montar Drive y clonar el repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_PATH = '/content/drive/MyDrive/tfg-datos'
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/monica793/tfg-datos.git {REPO_PATH}
else:
    !git -C {REPO_PATH} pull
print('Repo listo en', REPO_PATH)

In [ ]:
import sys

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print('cwd:', os.getcwd())

## 1. Dependencias

El E2E solo necesita **PyTorch** (Colab) y **NumPy**. **Sionna** solo en la sección opcional del híbrido.

In [ ]:
%pip install -q -U "numpy>=2.0,<2.3"

## 2. Cinco valores de `k` y entrenamiento secuencial E2E

Orden interno: para cada `k` se entrenan (o cargan) **todos** los `rho_db` antes de pasar al siguiente `k`.

In [ ]:
from utils.representative_ks import pick_representative_ks
from training.train_e2e import train_e2e_sweep_k, N, RHO_DBS

N_BLOCK = int(N)
REP_KS = pick_representative_ks(N_BLOCK, n_pick=5)
print('k representativos (espaciados, sin restricción Polar):', REP_KS)

e2e_by_rho = train_e2e_sweep_k(ks=REP_KS, n=N_BLOCK, rho_dbs=list(RHO_DBS))
print('Listo. Checkpoints: results/checkpoints/e2e_k<K>_n100_rho<R>.pt')

## 3. Curvas vs \(R=k/n\) (mismo estilo que `run_colab` / híbrido)

In [ ]:
from evaluation.plot_pfa_pmd_pie import run_curves_for_n
from systems.e2e_system import E2ESystem

for rho_db in RHO_DBS:

    def make_e2e(k, n, _rho=float(rho_db)):
        enc, dec = e2e_by_rho[_rho][k]
        return E2ESystem(
            k=k, n=n, encoder=enc, decoder=dec, p_empty=0.3, thresh=0.5
        )

    run_curves_for_n(
        n=N_BLOCK,
        rho_db=rho_db,
        make_system=make_e2e,
        label='End-to-End MLP (multi-k)',
        k_cand=REP_KS,
        polar_k_constraint=False,
        figure_path=f'results/figures/E2E_multik_n{N_BLOCK}_rho{rho_db}.png',
    )

## 4. (Opcional) Híbrido entrenado en los mismos `k` + comparativa

Instala Sionna y TensorFlow si hace falta. El AE se entrena en los `k` que sean **válidos para Polar 5G** dentro de `REP_KS` (`REP_KS_BOTH`), para comparar en los mismos puntos \(R=k/n\).

In [ ]:
try:
    import sionna.phy
    print('Sionna OK')
except ImportError:
    %pip install -q sionna

try:
    import tensorflow as tf
    print('TensorFlow OK', tf.__version__)
except ImportError:
    %pip install -q tensorflow

In [ ]:
from training.train_hybrid import (
    train_ae_for_n,
    N_FIXED,
    RHO_DBS as HYB_RHO,
    k_is_valid_for_5g,
)
from systems.hybrid_polar import ActivityAwarePolarSystem
from systems.e2e_system import E2ESystem
from evaluation.plot_pfa_pmd_pie import plot_comparison

assert int(N_FIXED) == N_BLOCK, 'Alinea N del híbrido con N_BLOCK del E2E'

REP_KS_BOTH = [k for k in REP_KS if k_is_valid_for_5g(k, int(N_FIXED))]
if set(REP_KS_BOTH) != set(REP_KS):
    print('Aviso: híbrido solo en k Polar-válidos:', REP_KS_BOTH)
if not REP_KS_BOTH:
    raise ValueError('Ningún k de REP_KS es válido para Polar5G(n); ajusta REP_KS.')

hybrid_models = {}
for rho_db in HYB_RHO:
    hybrid_models[rho_db] = train_ae_for_n(
        n=N_FIXED, rho_db=rho_db, valid_ks=REP_KS_BOTH
    )

for rho_db in HYB_RHO:
    ae = hybrid_models[rho_db]

    def make_hybrid(k, n, _ae=ae):
        return ActivityAwarePolarSystem(
            k=k, n=n, ae_model=_ae, p_empty=0.3, thresh=0.5
        )

    def make_e2e_cmp(k, n, _rho=float(rho_db)):
        enc, dec = e2e_by_rho[_rho][k]
        return E2ESystem(
            k=k, n=n, encoder=enc, decoder=dec, p_empty=0.3, thresh=0.5
        )

    systems = {
        'Híbrido Polar+AE': make_hybrid,
        'End-to-End MLP': make_e2e_cmp,
    }
    plot_comparison(
        n=N_FIXED,
        rho_db=rho_db,
        systems=systems,
        k_cand=REP_KS_BOTH,
        polar_k_constraint=True,
    )

## 5. (Opcional) Descargar checkpoints

Reutiliza la celda de ZIP de `run_colab.ipynb` si quieres bajar `.pt` y `.h5` desde `results/checkpoints/`.